# 一键修复：公网域名 502 / 8010 无监听 / ComfyUI 未启动

**用法**：把本笔记本上传到当前 Colab 会话（左侧文件窗格 → 上传到会话存储），打开后只运行下面这一个代码单元格。

它会自动完成：诊断端口 → 补依赖 → 拉起 ComfyUI(8188) → 写入 `/content/qwen_api.py` → 启动代理(8010) → 健康检查。
cloudflared 隧道进程保持不动，**原公网域名不变**，后端恢复后 502 自动消失。

> 根因：原笔记本里“写 qwen_api.py”那一格只是定义了 `app` 并直接 `uvicorn.run()`，
> 既触发 Colab 事件循环冲突，也从未把文件写入磁盘；后续启动代理去运行不存在的文件，
> 8010 从未监听，隧道后端为空 → 502。另外 ComfyUI 首次启动也超时，本 Cell 一并处理。

运行完请把输出贴回来，尤其是 `comfyui.log` 末尾日志，方便继续定位出图环节。


In [ ]:
"""Colab 会话内一键修复脚本。

502 根因：笔记本里"写 qwen_api.py"那一格其实只定义了 app 并直接 uvicorn.run()，
既触发 asyncio 事件循环冲突，也从未把 /content/qwen_api.py 写入磁盘；
后续启动代理去运行一个不存在的文件，8010 从未监听 → cloudflared 隧道后端为空 → 502。
另：ComfyUI(8188) 首次启动也可能因缺少 gguf 等依赖而失败。

本脚本按 诊断 → 补依赖 → 拉起 ComfyUI(8188) → 写 qwen_api.py → 启动 8010 → 健康检查 执行。
cloudflared 隧道进程保持不动，因此原公网域名不变，后端恢复后 502 自动消失。
"""
import os
import sys
import time
import socket
import subprocess
import json
import urllib.request


def port_open(p, host="127.0.0.1"):
    s = socket.socket()
    s.settimeout(1)
    try:
        s.connect((host, p))
        return True
    except Exception:
        return False
    finally:
        s.close()


def tail(path, n=40):
    if not os.path.exists(path):
        return "(无日志)"
    with open(path, errors="replace") as f:
        return "\n".join(f.read().splitlines()[-n:])


print("=" * 60)
print("诊断: 8188 (ComfyUI) =", port_open(8188), "| 8010 (FastAPI) =", port_open(8010))
print("--- /content/comfyui.log 末尾 30 行 ---")
print(tail("/content/comfyui.log", 30))
print("=" * 60)

# ---------- 1) 补依赖 ----------
print("\n[1/5] 补装 gguf / fastapi / uvicorn / requests ...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "gguf", "fastapi", "uvicorn", "requests"],
    check=False,
)
gguf_req = "/content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt"
if os.path.exists(gguf_req):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", gguf_req], check=False)

# ---------- 2) 拉起 ComfyUI ----------
print("\n[2/5] 检查 ComfyUI (8188) ...")
if port_open(8188):
    print("✅ ComfyUI 已在运行")
else:
    subprocess.run(
        "ps -eo pid,args | grep '[m]ain.py --listen' | awk '{print $1}' | xargs -r kill",
        shell=True,
    )
    time.sleep(1)
    log = open("/content/comfyui.log", "a")
    p = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", "8188"],
        cwd="/content/ComfyUI",
        stdout=log,
        stderr=log,
        start_new_session=True,
    )
    print("启动 ComfyUI PID:", p.pid)
    for _ in range(60):
        time.sleep(3)
        if port_open(8188):
            break
    if port_open(8188):
        print("✅ ComfyUI 已就绪: http://127.0.0.1:8188")
    else:
        print("❌ ComfyUI 仍未就绪，日志末尾 40 行:")
        print(tail("/content/comfyui.log", 40))

if port_open(8188):
    try:
        info = json.load(urllib.request.urlopen("http://127.0.0.1:8188/object_info", timeout=15))
        print("   UNETLoaderGGUF 节点已注册:", "UNETLoaderGGUF" in info)
    except Exception as e:
        print("   object_info 读取失败:", repr(e))

# ---------- 3) 写入 qwen_api.py（核心修复） ----------
print("\n[3/5] 写入 /content/qwen_api.py ...")
# qwen_api.py 完整源码（已内联，运行本 Cell 不需要联网下载任何脚本）
api_code = r'''# OpenAI 兼容代理：把 ComfyUI 工作流包装成 /v1/images/generations
# 若 ComfyUI-GGUF 节点类名不同，改 UNET_CLASS 即可
from fastapi import FastAPI
from pydantic import BaseModel
import requests, base64, uuid, time, uvicorn, traceback

app = FastAPI(title='Qwen-Image-2.1-Turbo proxy')
COMFY = 'http://127.0.0.1:8188'
UNET = 'qwen-image-2.1-turbo-Q4_K_M.gguf'
CLIP = 'qwen3vl_8b_int8_convrot.safetensors'
VAE = 'qwen_image_2.1_vae_bf16.safetensors'
UNET_CLASS = 'UNETLoaderGGUF'

class ImageGenRequest(BaseModel):
    prompt: str
    model: str = UNET
    width: int = 1024
    height: int = 1024
    steps: int = 8
    seed: int = -1
    cfg: float = 1.0

@app.get('/')
def root():
    return {'status': 'ok', 'backend': COMFY, 'endpoint': '/v1/images/generations'}

@app.get('/health')
def health():
    try:
        requests.get(COMFY + '/system_stats', timeout=5).raise_for_status()
        return {'ok': True, 'comfyui': 'up'}
    except Exception as e:
        return {'ok': False, 'comfyui': repr(e)}

@app.get('/v1/models')
def models():
    return {'object': 'list', 'data': [{'id': UNET, 'object': 'model'}]}

def build_workflow(req: ImageGenRequest):
    seed = req.seed if req.seed and req.seed >= 0 else uuid.uuid4().int >> 64
    return {
        '1': {'inputs': {'unet_name': req.model}, 'class_type': UNET_CLASS},
        '2': {'inputs': {'clip_name': CLIP, 'type': 'qwen_image'}, 'class_type': 'CLIPLoader'},
        '3': {'inputs': {'vae_name': VAE}, 'class_type': 'VAELoader'},
        '4': {'inputs': {'width': req.width, 'height': req.height, 'batch_size': 1}, 'class_type': 'EmptyLatentImage'},
        '5': {'inputs': {'model': ['1', 0], 'positive': ['2', 0], 'negative': ['2', 0], 'latent_image': ['4', 0],
                'seed': int(seed), 'steps': req.steps, 'cfg': req.cfg,
                'sampler_name': 'euler', 'scheduler': 'normal', 'denoise': 1.0}, 'class_type': 'KSampler'},
        '6': {'inputs': {'samples': ['5', 0], 'vae': ['3', 0]}, 'class_type': 'VAEDecode'},
        '7': {'inputs': {'images': ['6', 0], 'filename_prefix': 'qwen_colab'}, 'class_type': 'SaveImage'},
    }

@app.post('/v1/images/generations')
def generate(req: ImageGenRequest):
    try:
        r = requests.post(COMFY + '/prompt', json={'prompt': build_workflow(req)}, timeout=30)
        if r.status_code != 200:
            return {'error': 'comfyui /prompt rejected', 'detail': r.text[:1000]}
        pid = r.json()['prompt_id']
        outs = None
        for _ in range(180):
            time.sleep(2)
            h = requests.get(COMFY + f'/history/{pid}', timeout=10).json()
            if pid in h and 'outputs' in h[pid]:
                outs = h[pid]['outputs']
                break
        if outs is None:
            return {'error': 'generation timeout'}
        images = []
        for node_id, out in outs.items():
            for img in out.get('images', []):
                data = requests.get(COMFY + '/view',
                    params={'filename': img['filename'], 'subfolder': img.get('subfolder', ''), 'type': 'output'},
                    timeout=60).content
                images.append({'b64_json': base64.b64encode(data).decode('ascii')})
        return {'created': int(time.time()), 'data': images}
    except Exception:
        return {'error': 'proxy exception', 'detail': traceback.format_exc()[-1500:]}

if __name__ == '__main__':
    print('qwen_api: listening on 0.0.0.0:8010 -> ' + COMFY, flush=True)
    uvicorn.run(app, host='0.0.0.0', port=8010, log_level='info')
'''

with open("/content/qwen_api.py", "w", encoding="utf-8") as f:
    f.write(api_code)
print("✅ 已写入", len(api_code), "字节（来源：GitHub 修复版笔记本）")

# ---------- 4) 启动 8010 ----------
print("\n[4/5] 启动 FastAPI 代理 (8010) ...")
subprocess.run(
    "ps -eo pid,args | grep '[q]wen_api.py' | awk '{print $1}' | xargs -r kill",
    shell=True,
)
time.sleep(1)
api = subprocess.Popen(
    [sys.executable, "qwen_api.py"],
    cwd="/content",
    stdout=open("/content/qwen_api.log", "w"),
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print("FastAPI PID:", api.pid)
for _ in range(40):
    if port_open(8010):
        break
    time.sleep(2)
if port_open(8010):
    print("✅ 8010 已监听")
else:
    print("❌ 8010 未监听，日志:", tail("/content/qwen_api.log", 40))

# ---------- 5) 健康检查 ----------
print("\n[5/5] 健康检查 ...")
if port_open(8010):
    try:
        h = urllib.request.urlopen("http://127.0.0.1:8010/health", timeout=10).read().decode()
        print("   /health =", h)
    except Exception as e:
        print("   /health 失败:", repr(e))

n = subprocess.run(
    "ps -eo args | grep -c '[c]loudflared tunnel'", shell=True, capture_output=True, text=True
).stdout.strip()
print("   cloudflared 隧道进程数:", n, "(>0 表示原域名保留)")

print("\n" + "=" * 60)
if port_open(8010) and port_open(8188):
    print("✅ 后端已恢复，刷新原公网域名即可（502 应消失）")
    print("   https://ensemble-sql-doe-reserves.trycloudflare.com")
    print("   https://ensemble-sql-doe-reserves.trycloudflare.com/v1/images/generations")
else:
    print("⚠️ 仍有端口未就绪，请把上面的日志贴回来继续排查")
print("=" * 60)
